In [1]:
# Assign Noto fonts to Unicode character database

# Input:
#   unicode_character_database.pkl
#
# Outputs:
#   noto_font_table.pkl
#   unicode_to_notofonts_map.pkl

# Noto does not have every single Unicode glyph
# For another font system, replace:
#   - font discovery
#   - font-family filtering
#   - font metadata extraction
#   - file names


In [4]:
# Libraries

import os
import unicodedata
print("Unicode version:")
print(unicodedata.unidata_version)
from collections import Counter

import pandas as pd
import matplotlib.font_manager as fm
from fontTools.ttLib import TTFont, TTCollection
import pickle

Unicode version:
15.0.0


In [5]:
# Download Noto fonts for each Colab runtime

!apt-get update -qq
!apt-get install -y fonts-noto -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-noto-core.
(Reading database ... 118337 files and directories currently installed.)
Preparing to unpack .../0-fonts-noto-core_20201225-1build1_all.deb ...
Unpacking fonts-noto-core (20201225-1build1) ...
Selecting previously unselected package fonts-noto.
Preparing to unpack .../1-fonts-noto_20201225-1build1_all.deb ...
Unpacking fonts-noto (20201225-1build1) ...
Selecting previously unselected package fonts-noto-cjk.
Preparing to unpack .../2-fonts-noto-cjk_1%3a20220127+repack1-1_all.deb ...
Unpacking fonts-noto-cjk (1:20220127+repack1-1) ...
Selecting previously unselected package fonts-noto-cjk-extra.
Preparing to unpack .../3-fonts-noto-cjk-extra_1%3a20220127+repack1-1_all.deb ...
Unpacking fonts-noto-cjk-extra (1:20220127+repack1-1) ...
Selecting pre

In [6]:
# Choose location of input/output files:
# "drive"    = Google Drive
# "computer" = local machine

input_source = "drive"
output_source = "drive"


# Mount Google Drive if needed

if input_source == "drive" or output_source == "drive":

    from google.colab import drive

    drive.mount("/content/drive")


# Input file

if input_source == "drive":

    character_database_path = os.path.join(
        "/content/drive/MyDrive",
        "Character Complexity",
        "unicode_character_database.pkl"
    )

elif input_source == "computer":

    character_database_path = input(
        "Enter the full path to "
        "unicode_character_database.pkl: "
    ).strip()

else:

    raise ValueError(
        "input_source must be 'drive' or 'computer'."
    )


# Output files

if output_source == "drive":

    font_map_path = os.path.join(
        "/content/drive/MyDrive",
        "Character Complexity",
        "unicode_to_notofonts_map.pkl"
    )

    font_table_path = os.path.join(
        "/content/drive/MyDrive",
        "Character Complexity",
        "noto_font_table.pkl"
    )

elif output_source == "computer":

    output_directory = input(
        "Enter the directory where output files "
        "should be saved: "
    ).strip()

    font_map_path = os.path.join(
        output_directory,
        "unicode_to_notofonts_map.pkl"
    )

    font_table_path = os.path.join(
        output_directory,
        "noto_font_table.pkl"
    )

else:

    raise ValueError(
        "output_source must be 'drive' or 'computer'."
    )


print("Input:")
print(character_database_path)

print()
print("Outputs:")
print(font_map_path)
print(font_table_path)

Mounted at /content/drive
Input:
/content/drive/MyDrive/Character Complexity/unicode_character_database.pkl

Outputs:
/content/drive/MyDrive/Character Complexity/unicode_to_notofonts_map.pkl
/content/drive/MyDrive/Character Complexity/noto_font_table.pkl


In [7]:
# Load Unicode character database

if not os.path.exists(character_database_path):

    raise FileNotFoundError(
        f"Unicode character database not found:\n"
        f"{character_database_path}"
    )


character_database = pd.read_pickle(
    character_database_path
)


print(
    f"Unicode database loaded: "
    f"{len(character_database):,} characters"
)

Unicode database loaded: 146,550 characters


In [13]:
# Check database

display(character_database.head(20))
display(character_database.tail(20))

display(
    character_database["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

print()
print(
    "Minimum codepoint:",
    character_database["codepoint"].min()
)

print(
    "Maximum codepoint:",
    character_database["codepoint"].max()
)

print(
    f"Unique codepoints: "
    f"{character_database['codepoint'].nunique():,}"
)

print(
    f"Unique characters: "
    f"{character_database['char'].nunique():,}"
)

,codepoint,char,unicode,unicode_name,category
0,33,!,U+0021,EXCLAMATION MARK,Po
1,34,"""",U+0022,QUOTATION MARK,Po
2,35,#,U+0023,NUMBER SIGN,Po
3,36,$,U+0024,DOLLAR SIGN,Sc
4,37,%,U+0025,PERCENT SIGN,Po
5,38,&,U+0026,AMPERSAND,Po
6,39,',U+0027,APOSTROPHE,Po
7,40,(,U+0028,LEFT PARENTHESIS,Ps
8,41,),U+0029,RIGHT PARENTHESIS,Pe
9,42,*,U+002A,ASTERISK,Po


,codepoint,char,unicode,unicode_name,category
146530,205727,𲎟,U+3239F,CJK UNIFIED IDEOGRAPH-3239F,Lo
146531,205728,𲎠,U+323A0,CJK UNIFIED IDEOGRAPH-323A0,Lo
146532,205729,𲎡,U+323A1,CJK UNIFIED IDEOGRAPH-323A1,Lo
146533,205730,𲎢,U+323A2,CJK UNIFIED IDEOGRAPH-323A2,Lo
146534,205731,𲎣,U+323A3,CJK UNIFIED IDEOGRAPH-323A3,Lo
146535,205732,𲎤,U+323A4,CJK UNIFIED IDEOGRAPH-323A4,Lo
146536,205733,𲎥,U+323A5,CJK UNIFIED IDEOGRAPH-323A5,Lo
146537,205734,𲎦,U+323A6,CJK UNIFIED IDEOGRAPH-323A6,Lo
146538,205735,𲎧,U+323A7,CJK UNIFIED IDEOGRAPH-323A7,Lo
146539,205736,𲎨,U+323A8,CJK UNIFIED IDEOGRAPH-323A8,Lo


,category,count
0,Lo,131612
1,So,6634
2,Ll,2233
3,Lu,1831
4,Sm,948
5,No,915
6,Nd,680
7,Po,628
8,Lm,397
9,Nl,236



Minimum codepoint: 9
Maximum codepoint: 205743
Unique codepoints: 146,550
Unique characters: 146,550


In [14]:
# Find installed Noto font files.
# .ttf and .otf contain one font.
# .ttc contains multiple fonts in one file.

fonts = [
    f for f in fm.findSystemFonts()
    if "noto" in f.lower()
]

print(f"Noto font files found: {len(fonts):,}")
print()

# Check the types of font files found.
print("Font file types:")
print(Counter(os.path.splitext(f)[1].lower() for f in fonts))

Noto font files found: 2,394

Font file types:
Counter({'.ttf': 2380, '.ttc': 14})


In [20]:
# Build Noto Font Table and Unicode Codepoint to Font Map
#
# font_table:
#   one row per individual font
#   contains font name, family name, file path, and collection index

font_records = []
codepoint_to_fonts = {}

font_id = 0

for font_path in fonts:

    try:

        # Single font files (.ttf, .otf)
        if font_path.lower().endswith((".ttf", ".otf")):

            font = TTFont(font_path)

            cmap = font.getBestCmap()
            names = font["name"].names

            family_name = next(
                (
                    n.toUnicode()
                    for n in names
                    if n.nameID == 1
                ),
                ""
            )

            full_name = next(
                (
                    n.toUnicode()
                    for n in names
                    if n.nameID == 4
                ),
                family_name
            )

            font_records.append({
                "font_id": font_id,
                "font_name": full_name,
                "family_name": family_name,
                "font_path": font_path,
                "font_fileindex": 0
            })

            for codepoint in cmap:
                codepoint_to_fonts.setdefault(
                    codepoint, []
                ).append(font_id)

            font_id += 1


        # Font collections (.ttc)
        # for a .ttc you need font_path and font_fileindex within the file
        elif font_path.lower().endswith(".ttc"):

            collection = TTCollection(font_path)

            for index, font in enumerate(collection.fonts):

                cmap = font.getBestCmap()
                names = font["name"].names

                family_name = next(
                    (
                        n.toUnicode()
                        for n in names
                        if n.nameID == 1
                    ),
                    ""
                )

                full_name = next(
                    (
                        n.toUnicode()
                        for n in names
                        if n.nameID == 4
                    ),
                    family_name
                )

                font_records.append({
                    "font_id": font_id,
                    "font_name": full_name,
                    "family_name": family_name,
                    "font_path": font_path,
                    "font_fileindex": index
                })

                for codepoint in cmap:
                    codepoint_to_fonts.setdefault(
                        codepoint, []
                    ).append(font_id)

                font_id += 1

    except Exception:
        # Skip unreadable fonts
        pass


font_table = pd.DataFrame(font_records)

print(f"Individual fonts read: {len(font_table):,}")
print(f"Unicode codepoints with font support: {len(codepoint_to_fonts):,}")




Individual fonts read: 2,460
Unicode codepoints with font support: 79,029


In [21]:
# Inspect the fonts that were successfully read.
# This shows the actual font names rather than just file locations.

print(font_table[
    ["font_id", "font_name", "family_name", "font_fileindex"]
].head(20))
print(font_table[
    ["font_id", "font_name", "family_name", "font_fileindex"]
].tail(20))

# Count how many individual fonts came from each file type.

print(
    font_table["font_path"]
    .str.lower()
    .str.extract(r"(\.[^.]+)$")[0]
    .value_counts()
)

    font_id                                 font_name  \
0         0             Noto Sans Mono Condensed Bold   
1         1       Noto Sans Georgian Condensed Medium   
2         2                    Noto Sans Light Italic   
3         3      Noto Sans Canadian Aboriginal Medium   
4         4                Noto Looped Lao UI Regular   
5         5                   Noto Sans CJK JP Medium   
6         6                   Noto Sans CJK KR Medium   
7         7                   Noto Sans CJK SC Medium   
8         8                   Noto Sans CJK TC Medium   
9         9                   Noto Sans CJK HK Medium   
10       10     Noto Sans Thai UI SemiCondensed Light   
11       11         Noto Sans Psalter Pahlavi Regular   
12       12       Noto Sans Thai ExtraCondensed Light   
13       13                          Noto Serif Black   
14       14                   Noto Sans Display Black   
15       15  Noto Serif Ethiopic ExtraCondensed Light   
16       16                  No

In [23]:
# Create font ID -> font name map
# Creates a lookup dictionary from font_table so font IDs can be converted to human-readable font names when building unicode_to_notofonts

font_id_to_name = dict(
    zip(
        font_table["font_id"],
        font_table["font_name"]
    )
)


# Create Unicode to font ID + font name map
# unicode_to_notofonts: Unicode codepoint -> font IDs + font names that support it

unicode_to_notofonts = {
    codepoint: {
        "font_ids": ids,
        "font_names": [
            font_id_to_name[i]
            for i in ids
        ]
    }
    for codepoint, ids in codepoint_to_fonts.items()
}


# Assign fonts to Unicode database

character_database["font_ids"] = character_database["codepoint"].map(
    lambda x: codepoint_to_fonts.get(x, [])
)

character_database["font_count"] = character_database["font_ids"].apply(len)




In [37]:
# Inspect the first/last 20 characters and their font counts.
print(
    character_database[
        ["char", "unicode_name", "codepoint", "font_count"]
    ].head(20)
)
print(
    character_database[
        ["char", "unicode_name", "codepoint", "font_count"]
    ].tail(20)
)

   char       unicode_name  codepoint  font_count
0     !   EXCLAMATION MARK         33        1619
1     "     QUOTATION MARK         34        1525
2     #        NUMBER SIGN         35        1263
3     $        DOLLAR SIGN         36         410
4     %       PERCENT SIGN         37        1335
5     &          AMPERSAND         38         554
6     '         APOSTROPHE         39        1526
7     (   LEFT PARENTHESIS         40        1540
8     )  RIGHT PARENTHESIS         41        1540
9     *           ASTERISK         42        1412
10    +          PLUS SIGN         43        1268
11    ,              COMMA         44        1617
12    -       HYPHEN-MINUS         45        2013
13    .          FULL STOP         46        1621
14    /            SOLIDUS         47        1530
15    0         DIGIT ZERO         48        1360
16    1          DIGIT ONE         49        1360
17    2          DIGIT TWO         50        1360
18    3        DIGIT THREE         51        1360


In [34]:
# Check overall font coverage.

total = len(character_database)
renderable = (character_database["font_count"] > 0).sum()
missing_count = (character_database["font_count"] == 0).sum()

print(f"Total characters: {total:,}")
print(f"Characters with at least one font: {renderable:,}")
print(f"Characters with no font: {missing_count:,}")
print(f"Percent renderable: {100 * renderable / total:.2f}%")
print(f"Percent missing: {100 * missing_count / total:.2f}%")

Total characters: 146,550
Characters with at least one font: 76,816
Characters with no font: 69,734
Percent renderable: 52.42%
Percent missing: 47.58%


In [39]:
# Validate Unicode-to-font mapping
#
# First checks representative characters from different writing systems.
# Then displays all available fonts for the selected character.

examples = [
    "A",    # Latin
    "Ж",    # Cyrillic
    "中",   # CJK
    "अ",    # Devanagari
    "م",    # Arabic
    "Δ",    # Greek
    "ก",    # Thai
    "한",   # Korean
    "𗀀"    # Tangut
]

# Spot-check representative characters

spot_check = []

for char in examples:

    codepoint = ord(char)
    font_ids = codepoint_to_fonts.get(
        codepoint,
        []
    )

    spot_check.append({
        "character": char,
        "unicode": f"U+{codepoint:04X}",
        "unicode_name": unicodedata.name(
            char,
            "UNKNOWN"
        ),
        "font_count": len(font_ids)
    })

print("Representative character check")
print()

display(
    pd.DataFrame(spot_check)
)


# Inspect all fonts for one selected character

character = "中"

codepoint = ord(character)

font_ids = codepoint_to_fonts.get(
    codepoint,
    []
)

print()
print(
    f"Fonts available for "
    f"'{character}' "
    f"({unicodedata.name(character, 'UNKNOWN')})"
)

print(
    f"Number of fonts: "
    f"{len(font_ids):,}"
)

print()

display(
    font_table.loc[
        font_table["font_id"].isin(font_ids),
        [
            "font_id",
            "font_name",
            "family_name",
            "font_path",
            "font_fileindex"
        ]
    ].reset_index(drop=True)
)

Representative character check



,character,unicode,unicode_name,font_count
0,A,U+0041,LATIN CAPITAL LETTER A,416
1,Ж,U+0416,CYRILLIC CAPITAL LETTER ZHE,405
2,中,U+4E2D,CJK UNIFIED IDEOGRAPH-4E2D,80
3,अ,U+0905,DEVANAGARI LETTER A,108
4,م,U+0645,ARABIC LETTER MEEM,91
5,Δ,U+0394,GREEK CAPITAL LETTER DELTA,407
6,ก,U+0E01,THAI CHARACTER KO KAI,180
7,한,U+D55C,HANGUL SYLLABLE HAN,80
8,𗀀,U+17000,UNKNOWN,1



Fonts available for '中' (CJK UNIFIED IDEOGRAPH-4E2D)
Number of fonts: 80



,font_id,font_name,family_name,font_path,font_fileindex
0,5,Noto Sans CJK JP Medium,Noto Sans CJK JP Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,0
1,6,Noto Sans CJK KR Medium,Noto Sans CJK KR Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,1
2,7,Noto Sans CJK SC Medium,Noto Sans CJK SC Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,2
3,8,Noto Sans CJK TC Medium,Noto Sans CJK TC Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,3
4,9,Noto Sans CJK HK Medium,Noto Sans CJK HK Medium,/usr/share/fonts/opentype/noto/NotoSansCJK-Med...,4
...,...,...,...,...,...
75,2346,Noto Serif CJK JP,Noto Serif CJK JP,/usr/share/fonts/opentype/noto/NotoSerifCJK-Re...,0
76,2347,Noto Serif CJK KR,Noto Serif CJK KR,/usr/share/fonts/opentype/noto/NotoSerifCJK-Re...,1
77,2348,Noto Serif CJK SC,Noto Serif CJK SC,/usr/share/fonts/opentype/noto/NotoSerifCJK-Re...,2
78,2349,Noto Serif CJK TC,Noto Serif CJK TC,/usr/share/fonts/opentype/noto/NotoSerifCJK-Re...,3


In [42]:
# Show characters with the greatest number of available fonts.

print(
    character_database[
        ["char", "unicode_name", "category", "font_count"]
    ]
    .sort_values("font_count", ascending=False)
    .head(20)
    .to_string(index=False)
)


char                unicode_name category  font_count
                           SPACE       Zs        2459
   ◌               DOTTED CIRCLE       So        2290
   -                HYPHEN-MINUS       Pd        2013
   ‐                      HYPHEN       Pd        1915
   ?               QUESTION MARK       Po        1637
   “  LEFT DOUBLE QUOTATION MARK       Pi        1635
   ” RIGHT DOUBLE QUOTATION MARK       Pf        1635
   ‘  LEFT SINGLE QUOTATION MARK       Pi        1633
   ’ RIGHT SINGLE QUOTATION MARK       Pf        1633
   …         HORIZONTAL ELLIPSIS       Po        1627
   .                   FULL STOP       Po        1621
   !            EXCLAMATION MARK       Po        1619
   :                       COLON       Po        1619
   ,                       COMMA       Po        1617
   (            LEFT PARENTHESIS       Ps        1540
   )           RIGHT PARENTHESIS       Pe        1540
   ;                   SEMICOLON       Po        1532
   /                     SOL

In [47]:
# Show characters that currently have no available font.
#
# These remain in the Unicode database but cannot be rendered using the current Noto font collection.

missing_chars = character_database[
    character_database["font_count"] == 0
]

print(
    f"Characters without an available font: "
    f"{len(missing_chars):,}"
)

print()

columns = [
    "codepoint",
    "unicode",
    "unicode_name",
    "category",
    "char"
]

# First 20

print("First 20")
display(
    missing_chars[
        columns
    ].head(20)
)

# Last 20

print("Last 20")
display(
    missing_chars[
        columns
    ].tail(20)
)

Characters without an available font: 69,734

First 20


,codepoint,unicode,unicode_name,category,char
1254,1519,U+05EF,HEBREW YOD TRIANGLE,Lo,ׯ
1271,1565,U+061D,ARABIC END OF TEXT MARK,Po,؝
1709,2144,U+0860,SYRIAC LETTER MALAYALAM NGA,Lo,ࡠ
1710,2145,U+0861,SYRIAC LETTER MALAYALAM JA,Lo,ࡡ
1711,2146,U+0862,SYRIAC LETTER MALAYALAM NYA,Lo,ࡢ
1712,2147,U+0863,SYRIAC LETTER MALAYALAM TTA,Lo,ࡣ
1713,2148,U+0864,SYRIAC LETTER MALAYALAM NNA,Lo,ࡤ
1714,2149,U+0865,SYRIAC LETTER MALAYALAM NNNA,Lo,ࡥ
1715,2150,U+0866,SYRIAC LETTER MALAYALAM BHA,Lo,ࡦ
1716,2151,U+0867,SYRIAC LETTER MALAYALAM RA,Lo,ࡧ


Last 20


,codepoint,unicode,unicode_name,category,char
146527,205724,U+3239C,CJK UNIFIED IDEOGRAPH-3239C,Lo,𲎜
146528,205725,U+3239D,CJK UNIFIED IDEOGRAPH-3239D,Lo,𲎝
146529,205726,U+3239E,CJK UNIFIED IDEOGRAPH-3239E,Lo,𲎞
146530,205727,U+3239F,CJK UNIFIED IDEOGRAPH-3239F,Lo,𲎟
146531,205728,U+323A0,CJK UNIFIED IDEOGRAPH-323A0,Lo,𲎠
146532,205729,U+323A1,CJK UNIFIED IDEOGRAPH-323A1,Lo,𲎡
146533,205730,U+323A2,CJK UNIFIED IDEOGRAPH-323A2,Lo,𲎢
146534,205731,U+323A3,CJK UNIFIED IDEOGRAPH-323A3,Lo,𲎣
146535,205732,U+323A4,CJK UNIFIED IDEOGRAPH-323A4,Lo,𲎤
146536,205733,U+323A5,CJK UNIFIED IDEOGRAPH-323A5,Lo,𲎥


In [49]:
# Most missing characters by Unicode category

print("Missing characters by Unicode category")
display(
    missing_chars["category"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

Missing characters by Unicode category


,count
category,
Ll,78
Lm,138
Lo,69124
Lu,40
Nd,40
No,87
Pd,1
Pe,4
Po,42


In [50]:
# Save Unicode to Noto font map

if os.path.exists(font_map_path):

    response = input(
        f"{font_map_path} already exists. Overwrite? (y/n): "
    )

    if response.lower() != "y":
        raise RuntimeError("Save cancelled.")


with open(font_map_path, "wb") as f:

    pickle.dump(
        unicode_to_notofonts,
        f
    )


print(f"Saved: {font_map_path}")


Saved: /content/drive/MyDrive/Character Complexity/unicode_to_notofonts_map.pkl


In [51]:
# Save Noto font table

if os.path.exists(font_table_path):

    response = input(
        f"{font_table_path} already exists. Overwrite? (y/n): "
    )

    if response.lower() != "y":
        raise RuntimeError("Save cancelled.")


font_table.to_pickle(
    font_table_path
)


print(f"Saved: {font_table_path}")

Saved: /content/drive/MyDrive/Character Complexity/noto_font_table.pkl
